In [1]:
%pip install cassandra-driver

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("TP4-Cassandra-Spark")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

sc = spark.sparkContext
print("Version Spark :", spark.version)
print("Master        :", sc.master)
print("Parallélisme  :", sc.defaultParallelism)

Version Spark : 4.2.0
Master        : local[4]
Parallélisme  : 4


In [3]:
from cassandra.cluster import Cluster

In [4]:
cluster = Cluster(["cass1"])
session = cluster.connect()
print("Connexion Cassandra réussie")

Connexion Cassandra réussie


In [5]:
rows = session.execute("""
    SELECT keyspace_name
    FROM system_schema.keyspaces
""")
for row in rows :
    print(row.keyspace_name)

system_auth
system_schema
system_distributed
system
velib_cluster
system_traces


In [6]:
KEYSPACE = "velib_cluster"
TABLE = "stations_velib"

session.set_keyspace(KEYSPACE)

rows = session.execute("""
    SELECT column_name, type
    FROM system_schema.columns
    WHERE keyspace_name = 'velib_cluster'
        AND table_name = 'stations_velib'
""")
for row in rows : 
    print(row.column_name, "->", row.type)

bornes_disponibles -> int
capacite -> int
derniere_mise_a_jour -> timestamp
nom_station -> text
station_id -> text
vatiques_disponibles -> int


In [7]:
rows = session.execute("""
    SELECT *
    FROM stations_velib
    LIMIT 10
""")

for row in rows : 
    print(row)

Row(station_id='15047', bornes_disponibles=50, capacite=52, derniere_mise_a_jour=datetime.datetime(2026, 10, 7, 11, 26, 17), nom_station='Morillons - Dantzig', vatiques_disponibles=2)
Row(station_id='11104', bornes_disponibles=19, capacite=20, derniere_mise_a_jour=datetime.datetime(2026, 10, 7, 11, 25), nom_station='Charonne - Robert et Sonia Delaunay', vatiques_disponibles=1)
Row(station_id='9020', bornes_disponibles=19, capacite=21, derniere_mise_a_jour=datetime.datetime(2026, 10, 7, 11, 19, 35), nom_station='Toudouze - Clauzel', vatiques_disponibles=2)
Row(station_id='32017', bornes_disponibles=4, capacite=22, derniere_mise_a_jour=datetime.datetime(2026, 10, 7, 11, 28, 43), nom_station='Basilique', vatiques_disponibles=13)
Row(station_id='7003', bornes_disponibles=6, capacite=60, derniere_mise_a_jour=datetime.datetime(2026, 10, 7, 11, 29, 18), nom_station='Square Boucicaut', vatiques_disponibles=53)
Row(station_id='8026', bornes_disponibles=0, capacite=12, derniere_mise_a_jour=datet

In [8]:
rows = session.execute("""
    SELECT * 
    FROM stations_velib
""")

columns = rows.column_names
data = [tuple(row) for row in rows]

print("Colonnes :", columns)
print("Nombre de lignes récupérées :", len(data))

Colonnes : ['station_id', 'bornes_disponibles', 'capacite', 'derniere_mise_a_jour', 'nom_station', 'vatiques_disponibles']
Nombre de lignes récupérées : 22


In [9]:
df = spark.createDataFrame(data, columns)

df.show(10, truncate = False)
df.printSchema()
print("Nombre de ligne dans le DataFrame :", df.count())

+----------+------------------+--------+-----------------------+-----------------------------------+--------------------+
|station_id|bornes_disponibles|capacite|derniere_mise_a_jour   |nom_station                        |vatiques_disponibles|
+----------+------------------+--------+-----------------------+-----------------------------------+--------------------+
|15047     |50                |52      |2026-10-07 11:26:17    |Morillons - Dantzig                |2                   |
|11104     |19                |20      |2026-10-07 11:25:00    |Charonne - Robert et Sonia Delaunay|1                   |
|9020      |19                |21      |2026-10-07 11:19:35    |Toudouze - Clauzel                 |2                   |
|32017     |4                 |22      |2026-10-07 11:28:43    |Basilique                          |13                  |
|7003      |6                 |60      |2026-10-07 11:29:18    |Square Boucicaut                   |53                  |
|8026      |0           

Nombre de ligne dans le DataFrame : 22


### Données utilisées

- **Source :** table Cassandra `velib_cluster.stations_velib` (cluster 3 nœuds, RF = 3), lue avec `cassandra-driver` puis convertie en DataFrame Spark.
- **Contenu :** 20 stations importées depuis l'API Vélib' + 2 stations de test (`99999`, `99998`) écrites pendant les tests de panne du TP3. Ces 2 lignes sont exclues de l'analyse métier.
- **Colonnes importantes :** `capacite`, `vatiques_disponibles` (vélos disponibles), `bornes_disponibles` (places libres), avec `station_id` et `nom_station` pour identifier la station.
- **Information métier :** la disponibilité des vélos par station, pour repérer les stations **en tension** : presque vides (pas de vélo à emprunter) ou presque pleines (pas de place pour rendre son vélo).

In [10]:
from pyspark.sql import functions as F

In [11]:
df.select(
    "station_id",
    "nom_station",
    "capacite",
    "vatiques_disponibles",
    "bornes_disponibles"
).show(10,truncate=False)

+----------+-----------------------------------+--------+--------------------+------------------+
|station_id|nom_station                        |capacite|vatiques_disponibles|bornes_disponibles|
+----------+-----------------------------------+--------+--------------------+------------------+
|15047     |Morillons - Dantzig                |52      |2                   |50                |
|11104     |Charonne - Robert et Sonia Delaunay|20      |1                   |19                |
|9020      |Toudouze - Clauzel                 |21      |2                   |19                |
|32017     |Basilique                          |22      |13                  |4                 |
|7003      |Square Boucicaut                   |60      |53                  |6                 |
|8026      |Messine - Place Du Pérou           |12      |12                  |0                 |
|99999     |Station Test Panne                 |30      |15                  |15                |
|6108      |Saint-Ro

In [12]:
# Stations avec moins de 5 vélos disponibles
df.filter(F.col("vatiques_disponibles") < 5).show(10, truncate=False)
# Stations avec plus de 20 vélos disponibles
df.filter(F.col("vatiques_disponibles") > 20).show(10, truncate=False)

+----------+------------------+--------+--------------------+-----------------------------------+--------------------+
|station_id|bornes_disponibles|capacite|derniere_mise_a_jour|nom_station                        |vatiques_disponibles|
+----------+------------------+--------+--------------------+-----------------------------------+--------------------+
|15047     |50                |52      |2026-10-07 11:26:17 |Morillons - Dantzig                |2                   |
|11104     |19                |20      |2026-10-07 11:25:00 |Charonne - Robert et Sonia Delaunay|1                   |
|9020      |19                |21      |2026-10-07 11:19:35 |Toudouze - Clauzel                 |2                   |
|6108      |14                |17      |2026-10-07 11:27:32 |Saint-Romain - Cherche-Midi        |2                   |
+----------+------------------+--------+--------------------+-----------------------------------+--------------------+



+----------+------------------+--------+--------------------+--------------------------+--------------------+
|station_id|bornes_disponibles|capacite|derniere_mise_a_jour|nom_station               |vatiques_disponibles|
+----------+------------------+--------+--------------------+--------------------------+--------------------+
|7003      |6                 |60      |2026-10-07 11:29:18 |Square Boucicaut          |53                  |
|17026     |18                |40      |2026-10-07 11:28:44 |Jouffroy d'Abbans - Wagram|21                  |
|13101     |6                 |34      |2026-10-07 11:26:57 |Croulebarde - Corvisart   |28                  |
+----------+------------------+--------+--------------------+--------------------------+--------------------+



### Colonne calculée : taux de disponibilité

`taux_disponibilite = vatiques_disponibles / capacite × 100` : la part des places de la station occupées par un vélo disponible. Un taux proche de 0 % signifie une station vide, proche de 100 % une station pleine.

In [13]:
# On retire les 2 stations de test écrites pendant les pannes du TP3
STATIONS_TEST = ["99999", "99998"]
df_metier = df.filter(~F.col("station_id").isin(STATIONS_TEST))

df_taux = df_metier.withColumn(
    "taux_disponibilite",
    F.when(
        F.col("capacite") > 0,
        F.round(F.col("vatiques_disponibles") / F.col("capacite") * 100, 1)
    ).otherwise(None)
)

df_taux.select(
    "station_id",
    "nom_station",
    "capacite",
    "vatiques_disponibles",
    "bornes_disponibles",
    "taux_disponibilite"
).orderBy("taux_disponibilite").show(20, truncate=False)

print("Stations réelles analysées :", df_taux.count())

+----------+-----------------------------------+--------+--------------------+------------------+------------------+
|station_id|nom_station                        |capacite|vatiques_disponibles|bornes_disponibles|taux_disponibilite|
+----------+-----------------------------------+--------+--------------------+------------------+------------------+
|15047     |Morillons - Dantzig                |52      |2                   |50                |3.8               |
|11104     |Charonne - Robert et Sonia Delaunay|20      |1                   |19                |5.0               |
|9020      |Toudouze - Clauzel                 |21      |2                   |19                |9.5               |
|6108      |Saint-Romain - Cherche-Midi        |17      |2                   |14                |11.8              |
|17044     |Porte de Saint-Ouen - Bessières    |42      |5                   |37                |11.9              |
|17041     |Guersant - Gouvion-Saint-Cyr       |36      |5      

Stations réelles analysées : 20


### Actions et agrégations

Statistiques globales sur la flotte, puis nombre de stations par niveau de vélos disponibles (Faible < 5, Moyenne 5 à 15, Forte > 15).

In [14]:
df_taux.select(
    F.count("*").alias("nb_stations"),
    F.sum("capacite").alias("places"),
    F.sum("vatiques_disponibles").alias("velos"),
    F.sum("bornes_disponibles").alias("bornes_libres"),
    F.min("vatiques_disponibles").alias("minimum"),
    F.max("vatiques_disponibles").alias("maximum"),
    F.round(F.avg("vatiques_disponibles"), 1).alias("moyenne"),
    F.round(F.avg("taux_disponibilite"), 1).alias("taux_moyen")
).show()

+-----------+------+-----+-------------+-------+-------+-------+----------+
|nb_stations|places|velos|bornes_libres|minimum|maximum|moyenne|taux_moyen|
+-----------+------+-----+-------------+-------+-------+-------+----------+
|         20|   647|  259|          361|      1|     53|   13.0|      39.8|
+-----------+------+-----+-------------+-------+-------+-------+----------+



In [15]:
df_categorie = df_taux.withColumn(
    "categorie",
    F.when(F.col("vatiques_disponibles") < 5, "Faible")
     .when(F.col("vatiques_disponibles") <= 15, "Moyenne")
     .otherwise("Forte")
)

(df_categorie
    .groupBy("categorie")
    .agg(
        F.count("*").alias("nb_stations"),
        F.round(F.avg("vatiques_disponibles"), 1).alias("velos_moyens"),
        F.round(F.avg("taux_disponibilite"), 1).alias("taux_moyen")
    )
    .orderBy(F.desc("nb_stations"))
    .show())

+---------+-----------+------------+----------+
|categorie|nb_stations|velos_moyens|taux_moyen|
+---------+-----------+------------+----------+
|  Moyenne|         10|         9.8|      39.1|
|    Forte|          6|        25.7|      62.5|
|   Faible|          4|         1.8|       7.5|
+---------+-----------+------------+----------+



### Partitions

In [16]:
print("Nombre de partitions initiales :", df.rdd.getNumPartitions())
# Réorganisation explicite en 4 partitions
df4 = df.repartition(4) 
print("Nombre de partitions après repartition(4) :", df4.rdd.getNumPartitions())

Nombre de partitions initiales : 4


Nombre de partitions après repartition(4) : 4


In [17]:
df2 = df.repartition(2)
print("Nombre de partitions après repartition(2) :", df2.rdd.getNumPartitions())
print("Lignes par partition (df)  :", df.rdd.glom().map(len).collect())
print("Lignes par partition (df2) :", df2.rdd.glom().map(len).collect())

Nombre de partitions après repartition(2) : 2


Lignes par partition (df)  : [5, 5, 5, 7]


Lignes par partition (df2) : [11, 11]


### Lazy Evaluation

In [18]:
result = (
    df
    .filter(F.col("vatiques_disponibles")>5)
    .select("station_id", "nom_station", "vatiques_disponibles")
    .orderBy(F.desc("vatiques_disponibles"))
)

print("La plan logique est construit, mais aucun calcul n'a été exécuté.")

La plan logique est construit, mais aucun calcul n'a été exécuté.


In [19]:
result.show(10)

+----------+--------------------+--------------------+
|station_id|         nom_station|vatiques_disponibles|
+----------+--------------------+--------------------+
|      7003|    Square Boucicaut|                  53|
|     13101|Croulebarde - Cor...|                  28|
|     17026|Jouffroy d'Abbans...|                  21|
|     21108|    Mairie de Clichy|                  20|
|     15202|Bracion - Périphé...|                  16|
|     40011|   Bas du Mont-Mesly|                  16|
|     99999|  Station Test Panne|                  15|
|     33006|André Karman - Ré...|                  15|
|     99998|Station Test Panne 2|                  15|
|     32017|           Basilique|                  13|
+----------+--------------------+--------------------+
only showing top 10 rows


### Shuffle : plan d'exécution

Le `groupBy` oblige Spark à regrouper les lignes d'une même catégorie dans la même partition : on cherche l'opérateur `Exchange` dans le plan physique.

In [20]:
df_categorie.groupBy("categorie").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['categorie], ['categorie, 'count(1) AS count#242]
+- Project [station_id#0, bornes_disponibles#1L, capacite#2L, derniere_mise_a_jour#3, nom_station#4, vatiques_disponibles#5L, taux_disponibilite#89, CASE WHEN (vatiques_disponibles#5L < cast(5 as bigint)) THEN Faible WHEN (vatiques_disponibles#5L <= cast(15 as bigint)) THEN Moyenne ELSE Forte END AS categorie#190]
   +- Project [station_id#0, bornes_disponibles#1L, capacite#2L, derniere_mise_a_jour#3, nom_station#4, vatiques_disponibles#5L, CASE WHEN (capacite#2L > cast(0 as bigint)) THEN round(((cast(vatiques_disponibles#5L as double) / cast(capacite#2L as double)) * cast(100 as double)), 1) ELSE cast(null as double) END AS taux_disponibilite#89]
      +- Filter NOT station_id#0 IN (99999,99998)
         +- LogicalRDD [station_id#0, bornes_disponibles#1L, capacite#2L, derniere_mise_a_jour#3, nom_station#4, vatiques_disponibles#5L], false

== Analyzed Logical Plan ==
categorie: string, count: bigint

### Spark UI

On déclenche une agrégation (action avec shuffle), puis on lit les Jobs et les Stages dans l'API REST de la Spark UI, qui expose les mêmes chiffres que les onglets **Jobs** et **Stages** de http://localhost:4040.

In [21]:
import json
import urllib.request

def spark_ui(endpoint):
    url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/{endpoint}"
    with urllib.request.urlopen(url) as r:
        return json.load(r)

sc.setJobDescription("groupBy categorie (shuffle)")
df_categorie.groupBy("categorie").count().collect()
sc.setJobDescription(None)

jobs = sorted(spark_ui("jobs"), key=lambda j: j["jobId"])
print("Spark UI : port", sc.uiWebUrl.rsplit(":", 1)[1], "| Jobs exécutés :", len(jobs))
print()
for job in jobs[-2:]:
    print(f"Job {job['jobId']} | {job['name'][:45]:45} | stages {job['numCompletedStages']} "
          f"(+{job['numSkippedStages']} skipped) | tasks {job['numCompletedTasks']} "
          f"(+{job['numSkippedTasks']} skipped)")

Spark UI : port 4042 | Jobs exécutés : 24

Job 22 | collect at /tmp/ipykernel_38313/3846315695.py | stages 1 (+0 skipped) | tasks 4 (+0 skipped)
Job 23 | collect at /tmp/ipykernel_38313/3846315695.py | stages 1 (+1 skipped) | tasks 1 (+4 skipped)


In [22]:
stages = [s for s in spark_ui("stages") if s.get("description") == "groupBy categorie (shuffle)"]

print(f"{'Stage':>5} | {'Statut':8} | {'Tasks':>5} | {'Shuffle Write':>13} | {'Shuffle Read':>12} | Temps exécution")
for s in sorted(stages, key=lambda s: s["stageId"]):
    print(f"{s['stageId']:>5} | {s['status']:8} | {s['numTasks']:>5} | "
          f"{s['shuffleWriteBytes']:>11} o | {s['shuffleReadBytes']:>10} o | {s['executorRunTime']} ms")

Stage | Statut   | Tasks | Shuffle Write | Shuffle Read | Temps exécution
   27 | COMPLETE |     4 |         392 o |          0 o | 1385 ms
   29 | COMPLETE |     1 |           0 o |        392 o | 45 ms


## Analyse métier

**Question :** quelles stations sont en tension (risque d'être vides ou pleines), et combien de vélos faudrait-il déplacer ?

Seuils retenus sur le taux de disponibilité : **< 20 %** → risque vide, **> 80 %** → risque plein, sinon équilibrée.

Chaîne complète : Lecture Cassandra → DataFrame → Sélection → Filtre → Colonne calculée → Agrégation → Action.

In [23]:
# Lecture Cassandra -> DataFrame Spark
rows = session.execute(
    "SELECT station_id, nom_station, capacite, vatiques_disponibles, bornes_disponibles FROM stations_velib"
)
df_source = spark.createDataFrame([tuple(r) for r in rows], rows.column_names)

df_tension = (
    df_source
    # Sélection
    .select("station_id", "nom_station", "capacite", "vatiques_disponibles", "bornes_disponibles")
    # Filtre : vraies stations avec une capacité connue
    .filter(~F.col("station_id").isin(STATIONS_TEST) & (F.col("capacite") > 0))
    # Colonnes calculées
    .withColumn("taux_disponibilite",
                F.round(F.col("vatiques_disponibles") / F.col("capacite") * 100, 1))
    .withColumn("niveau_tension",
                F.when(F.col("taux_disponibilite") < 20, "Risque vide")
                 .when(F.col("taux_disponibilite") > 80, "Risque plein")
                 .otherwise("Équilibrée"))
)

# Agrégation
synthese = (
    df_tension
    .groupBy("niveau_tension")
    .agg(
        F.count("*").alias("nb_stations"),
        F.sum("capacite").alias("places"),
        F.sum("vatiques_disponibles").alias("velos"),
        F.sum("bornes_disponibles").alias("bornes_libres"),
        F.round(F.avg("taux_disponibilite"), 1).alias("taux_moyen")
    )
    .orderBy("taux_moyen")
)

# Action : c'est ici que Spark exécute toute la chaîne
synthese.show(truncate=False)

+--------------+-----------+------+-----+-------------+----------+
|niveau_tension|nb_stations|places|velos|bornes_libres|taux_moyen|
+--------------+-----------+------+-----+-------------+----------+
|Risque vide   |6          |188   |17   |166          |9.3       |
|Équilibrée    |11         |353   |149  |183          |42.7      |
|Risque plein  |3          |106   |93   |12           |90.2      |
+--------------+-----------+------+-----+-------------+----------+



In [24]:
# Stations à rééquilibrer en priorité
(df_tension
    .filter(F.col("niveau_tension") != "Équilibrée")
    .select("nom_station", "capacite", "vatiques_disponibles", "bornes_disponibles",
            "taux_disponibilite", "niveau_tension")
    .orderBy("taux_disponibilite")
    .show(truncate=False))

+-----------------------------------+--------+--------------------+------------------+------------------+--------------+
|nom_station                        |capacite|vatiques_disponibles|bornes_disponibles|taux_disponibilite|niveau_tension|
+-----------------------------------+--------+--------------------+------------------+------------------+--------------+
|Morillons - Dantzig                |52      |2                   |50                |3.8               |Risque vide   |
|Charonne - Robert et Sonia Delaunay|20      |1                   |19                |5.0               |Risque vide   |
|Toudouze - Clauzel                 |21      |2                   |19                |9.5               |Risque vide   |
|Saint-Romain - Cherche-Midi        |17      |2                   |14                |11.8              |Risque vide   |
|Porte de Saint-Ouen - Bessières    |42      |5                   |37                |11.9              |Risque vide   |
|Guersant - Gouvion-Saint-Cyr   

## Interprétation métier

- Le réseau n'est pas en pénurie : 259 vélos pour 647 places (taux moyen de 40 %) et 361 bornes libres. Le problème vient de la **répartition** des vélos, pas de leur nombre.
- 6 stations sur 20 risquent d'être vides : elles n'offrent que 17 vélos pour 188 places (9 %). Morillons - Dantzig n'a plus que 2 vélos pour 52 places.
- 3 stations sont presque pleines (90 % en moyenne, 12 bornes libres au total). Messine - Place du Pérou n'a plus aucune borne : les usagers ne peuvent pas y rendre leur vélo.
- Il faudrait apporter environ 24 vélos aux 6 stations vides pour les remonter à 20 %. 9 de ces vélos peuvent venir des 3 stations pleines, ce qui les ramènerait sous 80 %.
- Ces chiffres sont une photo de 20 stations à un instant donné (7 octobre, vers 11 h 30). Il faudrait des relevés réguliers sur tout le réseau pour confirmer ces tendances.